In [ ]:
import tifffile as tiff
import numpy as np
import matplotlib.pyplot as plt
import tifffile as tiff
import numpy as np
import pandas as pd
import os
import pyvista as pv
from PIL import Image
from mpl_toolkits.mplot3d import Axes3D
from model_3D_visualization import visualize_cone_pyvista, calculate_cone_radius, cropping_img
from Volume_bleeding import simulate_cone_insertion, find_first_black_pixel_slice, process_cone_positions
from Number_bleeding import vessel_seg, process_cone_positions_num


# Choose the files(path)
data_dir = 'mouse_data'          
assert os.path.isdir(data_dir), f"{os.path.abspath(data_dir)} not found - run this notebook from the CF_v2 folder"

# Reslice 2 and 3 are excluded: their raw data is too noisy.
tiff_files = [os.path.join(data_dir, f'Reslice of {i}.tif') for i in [0, 1, 4, 5, 6, 7, 8, 9]]

# Create folders to save the output
os.makedirs("output_csv", exist_ok=True)       
os.makedirs("output_images", exist_ok=True)

In [ ]:
# All Parameters Setting
x_center = 2200             # insertion postion(x)
y_center = 50               # insertion postion(y)
crop_margin_x = 80          # length for visualization
crop_margin_y = 50          # width for visualization
pos_num = 100               # numbers of insertions(for different positions)
output_img_type = "tif"     # png, tiff, jpg...
depth_limit = 1000

Electrodes_config = {
    "Carbon Fiber (8.4um)": {
        "tip_length": 160, "tip_radius_start": 3.4, "tip_radius_end": 0,
        "shank_length": 840, "shank_radius_start": 4.2, "shank_radius_end": 4.2
    },
    "Paradromics Connexus (20um)": {
        "tip_length": 125, "tip_radius_start": 10, "tip_radius_end": 0,
        "shank_length": 875, "shank_radius_start": 10, "shank_radius_end": 10
    },
    "Microprobes FMA (25um)": {
        "tip_length": 28.09, "tip_radius_start": 12.5, "tip_radius_end": 0,
        "shank_length": 971.91, "shank_radius_start": 12.5, "shank_radius_end": 12.5
    },
    "Neuralink Shuttle (25um)": {
        "tip_length": 10, "tip_radius_start": 12.5, "tip_radius_end": 12.5,
        "shank_length": 990, "shank_radius_start": 12.5, "shank_radius_end": 12.5
    },
    "Blackrock UEA (90um)": {
        "tip_length": 50, "tip_radius_start": 14, "tip_radius_end": 1.5,
        "shank_length": 950, "shank_radius_start": 45, "shank_radius_end": 14
    }
}

In [3]:
# Segment the vessel for all files

seg_save_dir = "./seg_cache"
os.makedirs(seg_save_dir, exist_ok=True)

for file_path in tiff_files:
    file_label = os.path.splitext(os.path.basename(file_path))[0]
    seg_path = os.path.join(seg_save_dir, f"{file_label}_seg.npz")

    if not os.path.exists(seg_path):
        print(f"Segmenting {file_label} ...")
        img_data = tiff.imread(file_path)
        img_data = np.transpose(img_data, axes=(0, 2, 1)).astype(np.uint16)
        seg_result = vessel_seg(img_data, min_size=2, connectivity=2, distance=2)
        np.savez_compressed(seg_path, seg=seg_result)

    else:
        print(f"Loading cached segmentation: {file_label}")
        seg_result = np.load(seg_path)['seg']


def load_segmented_data(file_path, seg_dir="./seg_cache"):
    file_label = os.path.splitext(os.path.basename(file_path))[0]
    seg_path = os.path.join(seg_dir, f"{file_label}_seg.npz")
    if not os.path.exists(seg_path):
        raise FileNotFoundError(f"Segmentation for {file_label} not found.")
    return np.load(seg_path)['seg']



Loading cached segmentation: Reslice of 0
Loading cached segmentation: Reslice of 1
Loading cached segmentation: Reslice of 4
Loading cached segmentation: Reslice of 5
Loading cached segmentation: Reslice of 6
Loading cached segmentation: Reslice of 7
Loading cached segmentation: Reslice of 8
Loading cached segmentation: Reslice of 9


In [ ]:
# Blackrock UEA (90um)

selected_electrode = "Blackrock UEA (90um)" #Adjust here for different electrodes
config = Electrodes_config[selected_electrode]

tip_length = config["tip_length"]
tip_radius_start = config["tip_radius_start"]
tip_radius_end = config["tip_radius_end"]
shank_length = config["shank_length"]
shank_radius_start = config["shank_radius_start"]
shank_radius_end = config["shank_radius_end"]

all_area_df = []
all_num_df = []

for file_path in tiff_files:
    print(f"Processing {file_path} ...")
    file_label = os.path.splitext(os.path.basename(file_path))[0]

    # Load
    img_data = tiff.imread(file_path)
    img_data = np.transpose(img_data, axes=(0, 2, 1)).astype(np.uint16)
    img_data_seg = load_segmented_data(file_path)


    # Calculate area and number
    areas = process_cone_positions(
        img_data, y_center, shank_length, shank_radius_start * 2, shank_radius_end * 2,
        tip_length, tip_radius_start * 2, tip_radius_end * 2, depth_limit, pos_num
    )
    nums = process_cone_positions_num(
        img_data_seg, y_center, shank_length, shank_radius_start * 2, shank_radius_end * 2,
        tip_length, tip_radius_start * 2, tip_radius_end * 2, depth_limit, pos_num
    )

    # Store results
    area_df = pd.DataFrame({"file": file_label, "position": list(range(pos_num)), "overlap_area": areas})
    num_df = pd.DataFrame({"file": file_label, "position": list(range(pos_num)), "overlap_number": nums})

    all_area_df.append(area_df)
    all_num_df.append(num_df)

    # 3D Visualization
    start_slice = find_first_black_pixel_slice(img_data, x_center, y_center)
    cropped_img_data, adjusted_x_center, adjusted_y_center = cropping_img(img_data, x_center, y_center, crop_margin_x, crop_margin_y, start_slice)

    plotter = visualize_cone_pyvista(
        cropped_img_data, adjusted_x_center, adjusted_y_center, shank_length=shank_length,
        shank_base_diameter=shank_radius_start * 2, shank_top_diameter=shank_radius_end * 2,
        tip_length=tip_length, tip_base_diameter=tip_radius_start * 2, tip_top_diameter=tip_radius_end * 2,
        start_slice=start_slice, depth_limit= depth_limit, ui= 0)

    img_output_name = f"{selected_electrode.replace(' ', '_').replace('(', '').replace(')', '')}_{file_label}.{output_img_type}"
    img_output_path = os.path.join("output_images", img_output_name)

    plotter.screenshot(img_output_path)
    img = Image.open(img_output_path)
    img.save(img_output_path, dpi=(300, 300))


# Save to CSV
volume_csv = os.path.join("output_csv", f"Volume_{selected_electrode.replace(' ', '_').replace('(', '').replace(')', '')}.csv")
number_csv = os.path.join("output_csv", f"Number_{selected_electrode.replace(' ', '_').replace('(', '').replace(')', '')}.csv")
pd.concat(all_area_df, ignore_index=True).to_csv(volume_csv, index=False) # Adjust the file name for each electrode
pd.concat(all_num_df, ignore_index=True).to_csv(number_csv, index=False) # Adjust the file name for each electrode

Processing mouse_data/Reslice of 0.tif ...
    Counting the volume of intersected vessels...
    Counting the number of intersected vessels...
Processing mouse_data/Reslice of 1.tif ...
    Counting the volume of intersected vessels...
    Counting the number of intersected vessels...
Processing mouse_data/Reslice of 4.tif ...
    Counting the volume of intersected vessels...
    Counting the number of intersected vessels...
Processing mouse_data/Reslice of 5.tif ...
    Counting the volume of intersected vessels...
    Counting the number of intersected vessels...
Processing mouse_data/Reslice of 6.tif ...
    Counting the volume of intersected vessels...
    Counting the number of intersected vessels...
Processing mouse_data/Reslice of 7.tif ...
    Counting the volume of intersected vessels...
    Counting the number of intersected vessels...
Processing mouse_data/Reslice of 8.tif ...
    Counting the volume of intersected vessels...


In [ ]:
# Neuralink Shuttle (25um)

selected_electrode = "Neuralink Shuttle (25um)" #Adjust here for different electrodes
config = Electrodes_config[selected_electrode]

tip_length = config["tip_length"]
tip_radius_start = config["tip_radius_start"]
tip_radius_end = config["tip_radius_end"]
shank_length = config["shank_length"]
shank_radius_start = config["shank_radius_start"]
shank_radius_end = config["shank_radius_end"]

all_area_df = []
all_num_df = []

for file_path in tiff_files:
    print(f"Processing {file_path} ...")
    file_label = os.path.splitext(os.path.basename(file_path))[0]

    # Load
    img_data = tiff.imread(file_path)
    img_data = np.transpose(img_data, axes=(0, 2, 1)).astype(np.uint16)
    img_data_seg = load_segmented_data(file_path)


    # Calculate area and number
    areas = process_cone_positions(
        img_data, y_center, shank_length, shank_radius_start * 2, shank_radius_end * 2,
        tip_length, tip_radius_start * 2, tip_radius_end * 2, depth_limit, pos_num
    )
    nums = process_cone_positions_num(
        img_data_seg, y_center, shank_length, shank_radius_start * 2, shank_radius_end * 2,
        tip_length, tip_radius_start * 2, tip_radius_end * 2, depth_limit, pos_num
    )

    # Store results
    area_df = pd.DataFrame({"file": file_label, "position": list(range(pos_num)), "overlap_area": areas})
    num_df = pd.DataFrame({"file": file_label, "position": list(range(pos_num)), "overlap_number": nums})

    all_area_df.append(area_df)
    all_num_df.append(num_df)

    # 3D Visualization
    start_slice = find_first_black_pixel_slice(img_data, x_center, y_center)
    cropped_img_data, adjusted_x_center, adjusted_y_center = cropping_img(img_data, x_center, y_center, crop_margin_x, crop_margin_y, start_slice)

    plotter = visualize_cone_pyvista(
        cropped_img_data, adjusted_x_center, adjusted_y_center, shank_length=shank_length,
        shank_base_diameter=shank_radius_start * 2, shank_top_diameter=shank_radius_end * 2,
        tip_length=tip_length, tip_base_diameter=tip_radius_start * 2, tip_top_diameter=tip_radius_end * 2,
        start_slice=start_slice, depth_limit= depth_limit, ui= 0)

    img_output_name = f"{selected_electrode.replace(' ', '_').replace('(', '').replace(')', '')}_{file_label}.{output_img_type}"
    img_output_path = os.path.join("output_images", img_output_name)

    plotter.screenshot(img_output_path)
    img = Image.open(img_output_path)
    img.save(img_output_path, dpi=(300, 300))


# Save to CSV
volume_csv = os.path.join("output_csv", f"Volume_{selected_electrode.replace(' ', '_').replace('(', '').replace(')', '')}.csv")
number_csv = os.path.join("output_csv", f"Number_{selected_electrode.replace(' ', '_').replace('(', '').replace(')', '')}.csv")
pd.concat(all_area_df, ignore_index=True).to_csv(volume_csv, index=False) # Adjust the file name for each electrode
pd.concat(all_num_df, ignore_index=True).to_csv(number_csv, index=False) # Adjust the file name for each electrode

Processing /Users/macbook/Desktop/Lab/CF_project/mouse data/Reslice of 0.tif ...
    Counting the volume of intersected vessels...
    Counting the number of intersected vessels...
Processing /Users/macbook/Desktop/Lab/CF_project/mouse data/Reslice of 1.tif ...
    Counting the volume of intersected vessels...
    Counting the number of intersected vessels...
Processing /Users/macbook/Desktop/Lab/CF_project/mouse data/Reslice of 4.tif ...
    Counting the volume of intersected vessels...
    Counting the number of intersected vessels...
Processing /Users/macbook/Desktop/Lab/CF_project/mouse data/Reslice of 5.tif ...
    Counting the volume of intersected vessels...
    Counting the number of intersected vessels...
Processing /Users/macbook/Desktop/Lab/CF_project/mouse data/Reslice of 6.tif ...
    Counting the volume of intersected vessels...
    Counting the number of intersected vessels...
Processing /Users/macbook/Desktop/Lab/CF_project/mouse data/Reslice of 7.tif ...
    Counting t

In [ ]:
# Microprobes FMA (25um)

selected_electrode = "Microprobes FMA (25um)" #Adjust here for different electrodes
config = Electrodes_config[selected_electrode]

tip_length = config["tip_length"]
tip_radius_start = config["tip_radius_start"]
tip_radius_end = config["tip_radius_end"]
shank_length = config["shank_length"]
shank_radius_start = config["shank_radius_start"]
shank_radius_end = config["shank_radius_end"]

all_area_df = []
all_num_df = []

for file_path in tiff_files:
    print(f"Processing {file_path} ...")
    file_label = os.path.splitext(os.path.basename(file_path))[0]

    # Load
    img_data = tiff.imread(file_path)
    img_data = np.transpose(img_data, axes=(0, 2, 1)).astype(np.uint16)
    img_data_seg = load_segmented_data(file_path)


    # Calculate area and number
    areas = process_cone_positions(
        img_data, y_center, shank_length, shank_radius_start * 2, shank_radius_end * 2,
        tip_length, tip_radius_start * 2, tip_radius_end * 2, depth_limit, pos_num
    )
    nums = process_cone_positions_num(
        img_data_seg, y_center, shank_length, shank_radius_start * 2, shank_radius_end * 2,
        tip_length, tip_radius_start * 2, tip_radius_end * 2, depth_limit, pos_num
    )

    # Store results
    area_df = pd.DataFrame({"file": file_label, "position": list(range(pos_num)), "overlap_area": areas})
    num_df = pd.DataFrame({"file": file_label, "position": list(range(pos_num)), "overlap_number": nums})

    all_area_df.append(area_df)
    all_num_df.append(num_df)

    # 3D Visualization
    start_slice = find_first_black_pixel_slice(img_data, x_center, y_center)
    cropped_img_data, adjusted_x_center, adjusted_y_center = cropping_img(img_data, x_center, y_center, crop_margin_x, crop_margin_y, start_slice)

    plotter = visualize_cone_pyvista(
        cropped_img_data, adjusted_x_center, adjusted_y_center, shank_length=shank_length,
        shank_base_diameter=shank_radius_start * 2, shank_top_diameter=shank_radius_end * 2,
        tip_length=tip_length, tip_base_diameter=tip_radius_start * 2, tip_top_diameter=tip_radius_end * 2,
        start_slice=start_slice, depth_limit= depth_limit, ui= 0)

    img_output_name = f"{selected_electrode.replace(' ', '_').replace('(', '').replace(')', '')}_{file_label}.{output_img_type}"
    img_output_path = os.path.join("output_images", img_output_name)

    plotter.screenshot(img_output_path)
    img = Image.open(img_output_path)
    img.save(img_output_path, dpi=(300, 300))


# Save to CSV
volume_csv = os.path.join("output_csv", f"Volume_{selected_electrode.replace(' ', '_').replace('(', '').replace(')', '')}.csv")
number_csv = os.path.join("output_csv", f"Number_{selected_electrode.replace(' ', '_').replace('(', '').replace(')', '')}.csv")
pd.concat(all_area_df, ignore_index=True).to_csv(volume_csv, index=False) # Adjust the file name for each electrode
pd.concat(all_num_df, ignore_index=True).to_csv(number_csv, index=False) # Adjust the file name for each electrode

Processing /Users/macbook/Desktop/Lab/CF_project/mouse data/Reslice of 0.tif ...
    Counting the volume of intersected vessels...
    Counting the number of intersected vessels...
Processing /Users/macbook/Desktop/Lab/CF_project/mouse data/Reslice of 1.tif ...
    Counting the volume of intersected vessels...
    Counting the number of intersected vessels...
Processing /Users/macbook/Desktop/Lab/CF_project/mouse data/Reslice of 4.tif ...
    Counting the volume of intersected vessels...
    Counting the number of intersected vessels...
Processing /Users/macbook/Desktop/Lab/CF_project/mouse data/Reslice of 5.tif ...
    Counting the volume of intersected vessels...
    Counting the number of intersected vessels...
Processing /Users/macbook/Desktop/Lab/CF_project/mouse data/Reslice of 6.tif ...
    Counting the volume of intersected vessels...
    Counting the number of intersected vessels...
Processing /Users/macbook/Desktop/Lab/CF_project/mouse data/Reslice of 7.tif ...
    Counting t

In [ ]:
# Paradromics Connexus (20um)

selected_electrode = "Paradromics Connexus (20um)" #Adjust here for different electrodes
config = Electrodes_config[selected_electrode]

tip_length = config["tip_length"]
tip_radius_start = config["tip_radius_start"]
tip_radius_end = config["tip_radius_end"]
shank_length = config["shank_length"]
shank_radius_start = config["shank_radius_start"]
shank_radius_end = config["shank_radius_end"]

all_area_df = []
all_num_df = []

for file_path in tiff_files:
    print(f"Processing {file_path} ...")
    file_label = os.path.splitext(os.path.basename(file_path))[0]

    # Load
    img_data = tiff.imread(file_path)
    img_data = np.transpose(img_data, axes=(0, 2, 1)).astype(np.uint16)
    img_data_seg = load_segmented_data(file_path)


    # Calculate area and number
    areas = process_cone_positions(
        img_data, y_center, shank_length, shank_radius_start * 2, shank_radius_end * 2,
        tip_length, tip_radius_start * 2, tip_radius_end * 2, depth_limit, pos_num
    )
    nums = process_cone_positions_num(
        img_data_seg, y_center, shank_length, shank_radius_start * 2, shank_radius_end * 2,
        tip_length, tip_radius_start * 2, tip_radius_end * 2, depth_limit, pos_num
    )

    # Store results
    area_df = pd.DataFrame({"file": file_label, "position": list(range(pos_num)), "overlap_area": areas})
    num_df = pd.DataFrame({"file": file_label, "position": list(range(pos_num)), "overlap_number": nums})

    all_area_df.append(area_df)
    all_num_df.append(num_df)

    # 3D Visualization
    start_slice = find_first_black_pixel_slice(img_data, x_center, y_center)
    cropped_img_data, adjusted_x_center, adjusted_y_center = cropping_img(img_data, x_center, y_center, crop_margin_x, crop_margin_y, start_slice)

    plotter = visualize_cone_pyvista(
        cropped_img_data, adjusted_x_center, adjusted_y_center, shank_length=shank_length,
        shank_base_diameter=shank_radius_start * 2, shank_top_diameter=shank_radius_end * 2,
        tip_length=tip_length, tip_base_diameter=tip_radius_start * 2, tip_top_diameter=tip_radius_end * 2,
        start_slice=start_slice, depth_limit= depth_limit, ui= 0)

    img_output_name = f"{selected_electrode.replace(' ', '_').replace('(', '').replace(')', '')}_{file_label}.{output_img_type}"
    img_output_path = os.path.join("output_images", img_output_name)

    plotter.screenshot(img_output_path)
    img = Image.open(img_output_path)
    img.save(img_output_path, dpi=(300, 300))


# Save to CSV
volume_csv = os.path.join("output_csv", f"Volume_{selected_electrode.replace(' ', '_').replace('(', '').replace(')', '')}.csv")
number_csv = os.path.join("output_csv", f"Number_{selected_electrode.replace(' ', '_').replace('(', '').replace(')', '')}.csv")
pd.concat(all_area_df, ignore_index=True).to_csv(volume_csv, index=False) # Adjust the file name for each electrode
pd.concat(all_num_df, ignore_index=True).to_csv(number_csv, index=False) # Adjust the file name for each electrode

Processing /Users/macbook/Desktop/Lab/CF_project/mouse data/Reslice of 0.tif ...
    Counting the volume of intersected vessels...
    Counting the number of intersected vessels...
Processing /Users/macbook/Desktop/Lab/CF_project/mouse data/Reslice of 1.tif ...
    Counting the volume of intersected vessels...
    Counting the number of intersected vessels...
Processing /Users/macbook/Desktop/Lab/CF_project/mouse data/Reslice of 4.tif ...
    Counting the volume of intersected vessels...
    Counting the number of intersected vessels...
Processing /Users/macbook/Desktop/Lab/CF_project/mouse data/Reslice of 5.tif ...
    Counting the volume of intersected vessels...
    Counting the number of intersected vessels...
Processing /Users/macbook/Desktop/Lab/CF_project/mouse data/Reslice of 6.tif ...
    Counting the volume of intersected vessels...
    Counting the number of intersected vessels...
Processing /Users/macbook/Desktop/Lab/CF_project/mouse data/Reslice of 7.tif ...
    Counting t

In [ ]:
# Carbon Fiber (8.4um)

selected_electrode = "Carbon Fiber (8.4um)" #Adjust here for different electrodes
config = Electrodes_config[selected_electrode]

tip_length = config["tip_length"]
tip_radius_start = config["tip_radius_start"]
tip_radius_end = config["tip_radius_end"]
shank_length = config["shank_length"]
shank_radius_start = config["shank_radius_start"]
shank_radius_end = config["shank_radius_end"]

all_area_df = []
all_num_df = []

for file_path in tiff_files:
    print(f"Processing {file_path} ...")
    file_label = os.path.splitext(os.path.basename(file_path))[0]

    # Load
    img_data = tiff.imread(file_path)
    img_data = np.transpose(img_data, axes=(0, 2, 1)).astype(np.uint16)
    img_data_seg = load_segmented_data(file_path)


    # Calculate area and number
    areas = process_cone_positions(
        img_data, y_center, shank_length, shank_radius_start * 2, shank_radius_end * 2,
        tip_length, tip_radius_start * 2, tip_radius_end * 2, depth_limit, pos_num
    )
    nums = process_cone_positions_num(
        img_data_seg, y_center, shank_length, shank_radius_start * 2, shank_radius_end * 2,
        tip_length, tip_radius_start * 2, tip_radius_end * 2, depth_limit, pos_num
    )

    # Store results
    area_df = pd.DataFrame({"file": file_label, "position": list(range(pos_num)), "overlap_area": areas})
    num_df = pd.DataFrame({"file": file_label, "position": list(range(pos_num)), "overlap_number": nums})

    all_area_df.append(area_df)
    all_num_df.append(num_df)

    # 3D Visualization
    start_slice = find_first_black_pixel_slice(img_data, x_center, y_center)
    cropped_img_data, adjusted_x_center, adjusted_y_center = cropping_img(img_data, x_center, y_center, crop_margin_x, crop_margin_y, start_slice)

    plotter = visualize_cone_pyvista(
        cropped_img_data, adjusted_x_center, adjusted_y_center, shank_length=shank_length,
        shank_base_diameter=shank_radius_start * 2, shank_top_diameter=shank_radius_end * 2,
        tip_length=tip_length, tip_base_diameter=tip_radius_start * 2, tip_top_diameter=tip_radius_end * 2,
        start_slice=start_slice, depth_limit= depth_limit, ui= 0)

    img_output_name = f"{selected_electrode.replace(' ', '_').replace('(', '').replace(')', '')}_{file_label}.{output_img_type}"
    img_output_path = os.path.join("output_images", img_output_name)

    plotter.screenshot(img_output_path)
    img = Image.open(img_output_path)
    img.save(img_output_path, dpi=(300, 300))


# Save to CSV
volume_csv = os.path.join("output_csv", f"Volume_{selected_electrode.replace(' ', '_').replace('(', '').replace(')', '')}.csv")
number_csv = os.path.join("output_csv", f"Number_{selected_electrode.replace(' ', '_').replace('(', '').replace(')', '')}.csv")
pd.concat(all_area_df, ignore_index=True).to_csv(volume_csv, index=False) # Adjust the file name for each electrode
pd.concat(all_num_df, ignore_index=True).to_csv(number_csv, index=False) # Adjust the file name for each electrode

Processing /Users/macbook/Desktop/Lab/CF_project/mouse data/Reslice of 0.tif ...
    Counting the volume of intersected vessels...
    Counting the number of intersected vessels...
Processing /Users/macbook/Desktop/Lab/CF_project/mouse data/Reslice of 1.tif ...
    Counting the volume of intersected vessels...
    Counting the number of intersected vessels...
Processing /Users/macbook/Desktop/Lab/CF_project/mouse data/Reslice of 4.tif ...
    Counting the volume of intersected vessels...
    Counting the number of intersected vessels...
Processing /Users/macbook/Desktop/Lab/CF_project/mouse data/Reslice of 5.tif ...
    Counting the volume of intersected vessels...
    Counting the number of intersected vessels...
Processing /Users/macbook/Desktop/Lab/CF_project/mouse data/Reslice of 6.tif ...
    Counting the volume of intersected vessels...
    Counting the number of intersected vessels...
Processing /Users/macbook/Desktop/Lab/CF_project/mouse data/Reslice of 7.tif ...
    Counting t